# Datakontroll — siste artikler

Henter artikler fra SQLite og leser tilhørende tekst fra Obsidian-vaulten.
Juster parameterne i neste celle.

In [1]:
# ── Parametere ────────────────────────────────────────────────
ANTALL      = 5        # Antall siste artikler å vise
SØKETEKST   = ""       # Filtrer på tittel/URL (tom = ingen filter)
ELEMENT_ID  = None     # Hent én spesifikk artikkel (None = bruk ANTALL)

In [2]:
import sqlite3
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Last .env fra repo-rot
repo_rot = Path("__file__").resolve().parent.parent
load_dotenv(repo_rot / ".env")

DB_STI    = repo_rot / os.getenv("DATABASE_STI", "data/monitor.db")
VAULT_ROT = Path(os.getenv("VAULT_ROT", ""))

print(f"Database : {DB_STI}  (finnes: {DB_STI.exists()})")
print(f"Vault    : {VAULT_ROT}  (finnes: {VAULT_ROT.exists()})")

Database : C:\Users\Audun\2026\claude\personalisert_monitor\data\monitor.db  (finnes: True)
Vault    : C:\Users\Audun\2026\claude\OBSIDIAN\monitor-evals  (finnes: True)


In [3]:
conn = sqlite3.connect(DB_STI)
conn.row_factory = sqlite3.Row

if ELEMENT_ID is not None:
    sql = "SELECT * FROM elementer WHERE id = ?"
    params = (ELEMENT_ID,)
elif SØKETEKST:
    sql = """
        SELECT * FROM elementer
        WHERE tittel LIKE ? OR url LIKE ?
        ORDER BY id DESC LIMIT ?
    """
    mønster = f"%{SØKETEKST}%"
    params = (mønster, mønster, ANTALL)
else:
    sql = "SELECT * FROM elementer ORDER BY id DESC LIMIT ?"
    params = (ANTALL,)

artikler = conn.execute(sql, params).fetchall()
print(f"Fant {len(artikler)} artikkel(er)")

Fant 1 artikkel(er)


In [4]:
for art in artikler:
    # ── Metadata ──────────────────────────────────────────────
    meta = dict(art)

    # Sammendrag (kan være tomt)
    sam = conn.execute(
        "SELECT tekst, prompt_versjon, opprettet FROM sammendrag WHERE element_id = ? ORDER BY id DESC LIMIT 1",
        (meta["id"],)
    ).fetchone()

    # Artikkeltekst fra vault
    tekst = "*(vault_sti er None)*"
    if meta["vault_sti"]:
        vault_fil = VAULT_ROT / meta["vault_sti"]
        if vault_fil.exists():
            tekst = vault_fil.read_text(encoding="utf-8")
        else:
            tekst = f"*(fil ikke funnet: {vault_fil})*"

    # ── Visning ───────────────────────────────────────────────
    display(Markdown(f"---\n## [{meta['tittel']}]({meta['url']})"))
    display(Markdown(
        f"| Felt | Verdi |\n|------|-------|\n"
        + "\n".join(
            f"| `{k}` | {v} |"
            for k, v in meta.items()
        )
    ))

    if sam:
        display(Markdown(f"**Sammendrag** *(prompt: {sam['prompt_versjon']}, {sam['opprettet']})*\n\n{sam['tekst']}"))
    else:
        display(Markdown("*Ingen sammendrag ennå*"))

    display(Markdown("### Artikkeltekst"))
    display(Markdown(tekst))

---
## [Predictions for the Future of RAG](https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/)

| Felt | Verdi |
|------|-------|
| `id` | 1 |
| `kilde_id` | 6 |
| `guid` | 86463fb6-3b46-4ec1-bfde-65e05f44629c |
| `url` | https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/ |
| `tittel` | Predictions for the Future of RAG |
| `publisert` | None |
| `hentet` | 2026-04-22T22:21:21.128352+00:00 |
| `vault_sti` | artikler\86463fb6-predictions-for-the-future-of-rag.md |
| `dead_letter` | 0 |

*Ingen sammendrag ennå*

### Artikkeltekst

---
element_id: 86463fb6-3b46-4ec1-bfde-65e05f44629c
url: https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/
kildetype: manuell
---

# Predictions for the Future of RAG

In the next 6 to 8 months, RAG will be used primarily for report generation. We'll see a shift from using RAG agents as question-answering systems to using them more as report-generation systems. This is because the value you can get from a report is much greater than the current RAG systems in use. I'll explain this by discussing what I've learned as a consultant about understanding value and then how I think companies should describe the value they deliver through RAG.

Rag is the feature, not the benefit.

If you want to learn more about I systematically improve RAG applications check out my free 6 email improving rag crash course

[Check out the free email course here](https://dub.link/6wk-rag-email)

## Reports over RAG

So why are reports better than RAG? Simply put, RAG systems suck because the value you derive is time saved from finding an answer. This is a one-dimensional value, and it's very hard to sell any value beyond that. Meanwhile, a report is a higher-value product because it is a decision-making tool that enables better resource allocation.

To better illustrate this, I'll give a couple of examples:

If I have one employee I'm paying hourly, they can use a RAG app to run a query, and then they can deliver an answer. This is a perfectly acceptable way of using RAG in one-dimensional static scenarios, such as asking single questions. However, when a research team wants to do interviews (question-answer queries), the deliverable isn't an answer to a set of questions. Instead, it's a report. So, the RAG app can save the time of 8 employees making 50 dollars an hour, whereas the report will cost $20,000. If the report is helping an executive allocate a 5million dollar budget, the price might charge becomes a much smaller portion of that investment? This is true even if the process to generate the report is just a RAG application in a for loop.

The value of these two items is communicated differently. RAG is evaluated as a percentage of wages, while the report is evaluated as a percentage of high-leverage outcomes.

Another way this plays out is if you're hiring. If you're interviewing a client with 6 rounds of interviews, you could use RAG to ask questions, which might work. What might be better is if your organization made a well-defined template on which you can make high-value decisions. Something like "Has this candidate worked in a team before", "Are they independent?", "Do they reflect our company's values?". These are pretty well-known and established parts of the hiring template.

If there were a service that could take this template and all the meeting notes from the six interviews and then generate a report for you and your team to review and utilize in your hiring process, the value would be derived from the decision-making and capital allocated to hire the candidate. A recruiter might take $40,000 on a $250,000 candidate, which means being able to make a better decision as a result of this hiring overview is enormous. The hypothetical hiring app's value is much greater than simple question-answer sets because the outcome of the RAG application is less clear than the outcome of having a high-quality report you can rely on to make key decisions for your business. This is because the end deliverable has a greater value that can be leveraged, even if the process is similar. A good interview panel knows what the question should be, but your hiring copilot should do more and help you get there.

## Why you need SOPs

Furthermore, how reports are written is incredibly important. Scaling decision-making and processes in a company often involves developing standard operating procedures (SOPs), which are a way of formatting various reports in a unified manner.

One of the reasons I attend workshops, get coaching, or read business books is because the outcome I am looking for is an SOP. For instance, I learned a way to write sales engagement letters that convert better. Now, all of my meetings fit this format and help make me far more money than the $40 dollar book I learned the template from cost. People are taught to give feedback and answer questions in specific ways. You get better outcomes when this output is structured correctly in something like a report or a template. Being able to pay for the right report template can be incredibly valuable because it ensures you're getting the outcome you actually need.

![SOP](https://media.geeksforgeeks.org/wp-content/uploads/20240419121413/Aim-of-SOP-(Standard-Operating-Procedure)-copy.webp)

I don't care so much about being able to read a chat transcript of a meeting I had. I care if I can turn that transcript into a format and report that I know will drive my desired business outcomes rather than just save me time. I want the AI to create a memo with clear deliverables for me or summarize the chat transcript to tell me, "This is the objective, this is how we make the decision, and here are the follow-ups."

Ultimately, a report's value goes beyond a wage worker answering questions—it supports high-leverage outcomes like strategic decision-making.

## Future outcomes

If RAG primarily becomes report generation it means two things are possible: 1. a marketplace of report-generating tools, and 2. the ability to effectively find the right report for your desired outcome. I think that question-answer sets are going to be of limited usefulness, while report generation addresses not only question-answer sets but the value of decision-making. When these reports are available in a marketplace of templates, they add further value because understanding what the template is defining becomes a skill in itself. These are the kinds of skills that people then take workshops on, get coaches for, and buy books about.

## Want to learn more?

I also wrote a 6 week email course on RAG, where I cover everything in my consulting work. It's free and you can:

[Check out the free email course here](https://dub.link/6wk-rag-email)

## Related Posts

For more insights on RAG systems and related topics, check out these posts:

- [The RAG Playbook](https://jxnl.co/writing/2024/08/19/rag-flywheel/) - A systematic approach to continually improve RAG systems
- [How to build a terrible RAG system](https://jxnl.co/writing/2024/01/07/inverted-thinking-rag/) - An inverted thinking exercise on RAG best practices
- [RAG is more than just embedding search](https://jxnl.co/writing/2023/09/17/rag-is-more-than-embeddings/) - Exploring advanced RAG techniques beyond simple embeddings
